# Chapter 2: Handling Images with PyTorch
**Module 02 – Intermediate Deep Learning with PyTorch**

> *Instructor: Michal Oleszak, Machine Learning Engineer*

## 2.1 What Is an Image?

An image consists of **pixels** ("picture elements"), each containing color information:

| Type | Representation |
|---|---|
| **Grayscale** | Single integer in range 0–255 per pixel |
| **Color (RGB)** | Three integers (R, G, B), each 0–255 per pixel |

For a 128×128 **color image**, the tensor shape is `(3, 128, 128)` — 3 channels × height × width.

## 2.2 Directory Structure for Image Datasets

PyTorch's `ImageFolder` expects a specific directory layout:

```
clouds_train/
  cumulus/
    img001.jpg
    img002.jpg
  cumulonimbus/
    img003.jpg
clouds_test/
  cumulus/
  cumulonimbus/
```

Each **subfolder** = one class. Images inside = samples for that class.

In [ ]:
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader

# Define image transformations
train_transforms = transforms.Compose([
    transforms.ToTensor(),            # Convert PIL image to tensor [0,1]
    transforms.Resize((128, 128)),    # Resize all images to 128×128
    transforms.RandomHorizontalFlip(), # Data augmentation: random flip
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   # ImageNet mean
        std=[0.229, 0.224, 0.225]     # ImageNet std
    )
])

# Create dataset
# dataset_train = ImageFolder(
#     root='_dataset_c2/clouds_train',
#     transform=train_transforms
# )
# dataloader_train = DataLoader(dataset_train, batch_size=32, shuffle=True)
print("ImageFolder pipeline defined successfully.")

## 2.3 Convolutional Neural Networks (CNNs)

CNNs are the go-to architecture for image data:

| Layer | Purpose |
|---|---|
| `Conv2d` | Learns spatial features (edges, textures, shapes) |
| `MaxPool2d` | Reduces spatial dimensions (downsampling) |
| `Flatten` | Converts 2D feature maps to 1D vector |
| `Linear` | Classification head |

In [ ]:
import torch
import torch.nn as nn

class CloudClassifier(nn.Module):
    def __init__(self, num_classes=4):
        super(CloudClassifier, self).__init__()

        # Feature extractor
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 3 channels in, 32 out
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # 128 -> 64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # 64 -> 32
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # 32 -> 16
        )

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CloudClassifier(num_classes=4)
print(model)

# Test with a dummy batch: 4 images of 3x128x128
dummy_input = torch.randn(4, 3, 128, 128)
output = model(dummy_input)
print(f"\nOutput shape: {output.shape}  (batch=4, classes=4)")

## 2.4 Displaying Images from a DataLoader

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def imshow(img_tensor, title=None):
    """Display a tensor as an image."""
    img = img_tensor.numpy().transpose((1, 2, 0))  # C,H,W -> H,W,C
    img = np.clip(img * 0.5 + 0.5, 0, 1)  # Denormalize
    plt.imshow(img)
    if title:
        plt.title(title)
    plt.axis('off')

# Simulate a batch of random images
batch = torch.randn(4, 3, 64, 64)
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    plt.sca(ax)
    imshow(batch[i])
    plt.title(f"Image {i+1}")
plt.tight_layout()
plt.show()

## Summary

| Component | Key Detail |
|---|---|
| Pixel | Smallest image unit; grayscale = 1 int, RGB = 3 ints |
| `ImageFolder` | Auto-loads images from folder structure |
| `transforms.Compose` | Chain multiple preprocessing steps |
| CNN | Best architecture for image data |